In [ ]:
import pandas as pd

In [ ]:
merged_df = pd.read_csv("../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/all_user_windows_7d.csv")

In [ ]:
merged_df

In [ ]:
import os
import requests
import json
import time
import re
import random
import csv
import threading
from collections import defaultdict

# ------------------------------------------------------------
# 0. load checkpoint
# ------------------------------------------------------------

checkpoint_file = "../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/checkpoint_7d.json"


def load_checkpoint():
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"last_index": -1}


def save_checkpoint(last_index):
    with open(checkpoint_file, "w", encoding="utf-8") as f:
        json.dump({"last_index": last_index}, f)


# ------------------------------------------------------------
# 1. Endpoint & authentication
# ------------------------------------------------------------
endpoint_base = "https://aipolling.services.ai.azure.com/models"
api_version = "2024-05-01-preview"
url = f"{endpoint_base}/chat/completions?api-version={api_version}"

api_key = os.getenv("AZURE_API_KEY")
if not api_key:
    raise ValueError("AZURE_API_KEY environment variable is not set")


# ------------------------------------------------------------
# 2. Load belief categories from JSON
# ------------------------------------------------------------
with open("../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/updated_belief_features.json", "r", encoding="utf-8") as f:
    feature_data = json.load(f)

demo_features = feature_data["demo"]
belief_features = feature_data["beliefs"]


# ------------------------------------------------------------
# 3. Base prompt
# ------------------------------------------------------------
prompt_template = """
A social media account has the following username, name, description:
- Name: {name}
- Username: {username}
- Description: {description}
A structured description of their profile image is the following:
- Profile Image Description: {profile_image_description}
Furthermore, they self-report their location in their bio as: {location}
The amount of followers they have is: {followers}
They have written the following tweet(s): {tweets}

Below is a list of survey questions; each question is followed by a set of options, each labeled with a symbol that starts with that question's own id (e.g. "IMMCRIME1_1", "IMMCRIME1_2", etc.) -- some questions below have very similar wording to each other, so always double-check that the id prefix on the symbol/option you're using matches the question you're currently answering.
Please select, for each question, the most likely option for this user.

For each selected option/symbol, rate the speculation level (0-100), using the following scale: 
- 0-20: Explicit, direct evidence (e.g., stated in profile/tweets)
- 21-40: Strong indirect indicators
- 41-60: Partial hints or mixed evidence
- 61-80: Weak, limited indicators
- 81-100: No meaningful evidence

Your answer should pertain only to the information available in the user data -- namely the username, name, description, location, amount of followers, profile picture and tweets from this user -- and should not be affected by additional information available to you from any other source.

The survey questions are:
"""


# ------------------------------------------------------------
# 4. Build structured-output schema and prompt helpers
# ------------------------------------------------------------
# Each feature in updated_belief_features.json now looks like:
#   {"id": "<matching-target id, e.g. a GSS/ANES/CCES codebook variable>",
#    "feature_name": "<question text shown in the prompt>",
#    "options": ["<category 1 text>", "<category 2 text>", ...]}
# `id` is the single source of truth for "which question is this" -- it's used
# as the JSON schema property key and as the CSV column-name prefix.

# generate json schema for one belief/demo feature
def build_assessment_schema(feature):
    num_options = len(feature["options"])
    # symbols are tagged with this feature's own id (e.g. "IMMCRIME1_1",
    # "IMMCRIME1_2", ...) so they're globally unique across the whole prompt,
    # instead of a bare "1"/"2"/"3" that looks identical across ~40
    # near-duplicate questions in the same call. This gives the model a
    # harder-to-confuse anchor when two questions have very similar wording
    # (e.g. IMMCRIME1 vs IMMJOBS1, both opening with the same "There are
    # different opinions about immigrants..." boilerplate) -- observed
    # directly in a test run swapping answers between exactly that pair.
    # parse_structured_response() strips the "{id}_" prefix back off before
    # persisting, so the CSV's _Symbol column keeps its existing plain "1"/
    # "2"/... ordinal format for downstream consumers.
    symbol_options = [f"{feature['id']}_{i + 1}" for i in range(num_options)]
    category_options = list(feature["options"])  # the option texts themselves

    return {
        "type": "object",
        "properties": {
            "title": { # must match the feature_name exactly
                "type": "string",
                "enum": [feature["feature_name"]],
            }, 
            "symbol": { # this feature's id + 1-based position within `options`
                "type": "string",
                "enum": symbol_options,
            },
            "category": { # must be one of the option texts
                "type": "string",
                "enum": category_options,
            },
            "speculation": { # must be an integer between 0 and 100
                "type": "integer",
                "minimum": 0,
                "maximum": 100,
            },
        },
        "required": ["title", "symbol", "category", "speculation"],
        "additionalProperties": False,
    }

# generate json schema for all belief/demo features,
# for example: {
#     "name": "demo_assessments",
#     "strict": True,
#     "schema": {
#         "type": "object",
#         "properties": {
#             "M": {  # this feature's "id" (e.g. MARITAL STATUS's matching-target code)
#                 "type": "object",
#                 "properties": {
#                     "title": {"type": "string", "enum": ["MARITAL STATUS"]},
#                     "symbol": {"type": "string", "enum": ["M_1", "M_2"]},
#                     "category": {"type": "string", "enum": ["married...", "single..."]},
#                     "speculation": {"type": "integer"},
#                 },
#                 "required": ["title", "symbol", "category", "speculation"],
#                 "additionalProperties": False,
#             },
#             "Pid": {  # PARTISAN LOYALTIES's id
#                 "type": "object",
#                 "properties": {
#                     # same structure
#                 },
#                 "required": [...],
#                 "additionalProperties": False,
#             },
#         },
#         "required": ["M", "Pid"],
#         "additionalProperties": False,
#     },
# }
def build_response_schema(features, schema_name):
    properties = {}
    required = []

    for feature in features:
        key = feature["id"]
        properties[key] = build_assessment_schema(feature)
        required.append(key)

    return {
        "name": schema_name,
        "strict": True, # model output must strictly adhere to the schema, no extra fields allowed
        "schema": {
            "type": "object",
            "properties": properties,
            "required": required,
            "additionalProperties": False,
        },
    }


def build_belief_prompt(user_row, beliefs):
    # `beliefs` is expected to already be in the order the caller wants
    # shown/answered in -- shuffling happens once in infer_beliefs() so the
    # same order is used here and in build_response_schema(), instead of
    # shuffling again locally and silently diverging from the schema's key
    # order (see infer_beliefs()).

    # for example:
    # - OPINION ON THE STATEMENT << IMMIGRANTS INCREASE CRIME RATES >>: Choose one of [IMMCRIME1_1) Strongly agree with the statment << Immigrants increase crime rates >>, IMMCRIME1_2) Somewhat agree with the statment << Immigrants increase crime rates >>, IMMCRIME1_3) Somewhat disagree with the statment << Immigrants increase crime rates >>, IMMCRIME1_4) Strongly disagree with the statment << Immigrants increase crime rates >>]
    # - OPINION ON THE NUMBER OF IMMIGRANTS IN AMERICA: Choose one of [LETIN1A_1) The number of immigrants in the USA should be increased a lot, LETIN1A_2) The number of immigrants in the USA should be increased a little, LETIN1A_3) The number of immigrants in the USA should remain the same as it is, LETIN1A_4) The number of immigrants in the USA should be reduced a little, LETIN1A_5) The number of immigrants in the USA should be reduced a lot]

    lines = []
    for belief in beliefs:
        option_parts = [
            f"{belief['id']}_{i + 1}) {text}"
            for i, text in enumerate(belief['options'])
        ]
        lines.append(f"- {belief['feature_name']}: Choose one of [{', '.join(option_parts)}]")
    categories_text = "\n".join(lines)

    return prompt_template.format(
        name=user_row['name'],
        username=user_row['username'],
        description=user_row['description'],
        location=user_row['location'],
        followers=user_row['public_metrics.followers_count'],
        tweets=user_row['tweets_struct'],
        profile_image_description=user_row['profile_image_main_info'],
    ) + categories_text


# ------------------------------------------------------------
# 5. Structured response parsing
# ------------------------------------------------------------
def parse_structured_response(response_payload, features, prefix=None):
    parsed_values = {}

    for feature in features:
        key = feature["id"]
        output_key = f"{prefix}_{key}" if prefix else key

        if key not in response_payload:
            raise ValueError(f"Missing structured field: {key}")

        assessment = response_payload[key]
        if not isinstance(assessment, dict):
            raise ValueError(f"Structured field {key} is not an object")

        # The schema declares `title` as a single-value enum equal to this
        # feature's own feature_name, but that constraint isn't reliably
        # enforced by the backend for this model -- observed directly in a
        # test run, where several keys came back holding a *different*
        # question's title/category (e.g. CAPPUN's answer was actually
        # GUNLAW's). Treat that as a failed attempt so it retries instead of
        # silently persisting an answer to the wrong question.
        returned_title = assessment.get("title")
        if returned_title != feature["feature_name"]:
            raise ValueError(
                f"Title mismatch for {key}: expected {feature['feature_name']!r}, got {returned_title!r}"
            )

        # `symbol` is tagged with this feature's own id in the schema/prompt
        # (e.g. "IMMCRIME1_1") -- see build_assessment_schema(). Strip the
        # prefix back off before persisting, so the CSV's _Symbol column
        # keeps its existing plain "1"/"2"/... ordinal format for downstream
        # consumers (many MrP_pipeline scripts read *_Symbol as a plain
        # ordinal index). A prefix that doesn't match this key is the same
        # failure mode as a title mismatch -- treat it the same way.
        returned_symbol = assessment.get("symbol")
        expected_prefix = f"{key}_"
        if not isinstance(returned_symbol, str) or not returned_symbol.startswith(expected_prefix):
            raise ValueError(
                f"Symbol mismatch for {key}: expected prefix {expected_prefix!r}, got {returned_symbol!r}"
            )
        plain_symbol = returned_symbol[len(expected_prefix):]

        parsed_values[f"{output_key}_Category"] = assessment.get("category")
        parsed_values[f"{output_key}_Symbol"] = plain_symbol
        parsed_values[f"{output_key}_Speculation"] = assessment.get("speculation")

    return parsed_values # e.g., {"Belief_LETIN1A_Category": "Agree", "Belief_LETIN1A_Symbol": "1", ...}


# ------------------------------------------------------------
# 6. Build the request body
# ------------------------------------------------------------

class ContextLengthExceededError(Exception):
    """Raised when the request payload exceeds the model's context window.
    Confirmed real Azure error body (2026-08-24 diagnostic run) looks like:
    {"error":{"code":"Bad Request","status":400,
      "message":"{\\"object\\":\\"error\\",\\"message\\":\\"The input (180081 tokens) is longer
      than the model's context length (131072 tokens).\\",\\"type\\":\\"BadRequestError\\",
      \\"param\\":null,\\"code\\":400}"}}
    There is no dedicated semantic error code (outer/inner code are both generic
    "Bad Request"/400) -- detection has to match on the message text itself.

    A second variant (seen in the 2026-10-23 pilot) means the input itself
    fits, and only input + max_tokens doesn't:
      "Requested token count exceeds the model's maximum context length of
      131072 tokens. You requested a total of 132408 tokens: 116024 tokens from
      the input messages and 16384 tokens for the completion. ..."
    infer_beliefs_with_retry() retries that variant once with a smaller
    max_tokens -- see parse_input_tokens().
    """
    pass


CONTEXT_LIMIT_TOKENS = 131072   # gpt-oss-120b context window
MAX_COMPLETION_TOKENS = 16384   # default completion budget (reasoning + JSON answer)
COMPLETION_MARGIN_TOKENS = 256  # slack when shrinking the budget to fit -- input counts can shift slightly between shuffles


def parse_input_tokens(error_text):
    # input token count from the "input + completion too long" variant, or
    # None for the "input alone too long" variant (nothing to rescue there)
    match = re.search(r"(\d+) tokens from the input messages", error_text)
    return int(match.group(1)) if match else None


def min_completion_tokens(features):
    # smallest budget worth retrying with: observed output in the 2026-10-23
    # pilot was ~4.9K tokens median / 11K max for the 40 belief questions and
    # ~1.3K / 2K for the 10 demo ones -- below this, the answer would likely be
    # cut off and every retry wasted, so the row is skipped instead
    return 200 * len(features) + 1000


class RateLimitedError(Exception):
    """Raised on HTTP 429. Says nothing about the row itself -- the deployment
    (5M TPM / 5K RPM) is just momentarily over its limit, which becomes likely
    once the production loop runs many threads at once. infer_beliefs_with_retry()
    waits `retry_after` seconds and retries without using up one of its attempts.
    """
    def __init__(self, retry_after):
        super().__init__(f"rate limited, retry after {retry_after}s")
        self.retry_after = retry_after


def parse_retry_after(response, default=10.0):
    # Retry-After is usually seconds, but may be missing or an HTTP date
    try:
        return float(response.headers.get("Retry-After", default))
    except (TypeError, ValueError):
        return default


# Network-level failures (no connection, dropped connection, no response
# within the request timeout). Like a 429 these say nothing about the row --
# e.g. the local network dropping mid-pilot (2026-10-23) -- so they are
# retried without using up an attempt, instead of risking a row being
# persisted as exhausted_retries during an outage (key-based resume would
# then treat it as done and never re-run it).
NETWORK_ERRORS = (
    requests.exceptions.ConnectionError,       # includes ConnectTimeout
    requests.exceptions.Timeout,               # includes ReadTimeout
    requests.exceptions.ChunkedEncodingError,  # connection broken mid-response
)
NETWORK_WAIT_MAX = 300  # cap on the wait between network retries, seconds


# total 429s / network errors seen, shared across threads -- reset and printed by the production cell
rate_limit_lock = threading.Lock()
rate_limit_hits = 0
network_error_hits = 0


def record_rate_limit_hit():
    global rate_limit_hits
    with rate_limit_lock:
        rate_limit_hits += 1


def record_network_error_hit():
    global network_error_hits
    with rate_limit_lock:
        network_error_hits += 1


def infer_beliefs(row, model_name, features, schema_name, max_tokens=MAX_COMPLETION_TOKENS):
    # Shuffle once here, and use the *same* shuffled order for both the
    # prompt text and the response schema's property order -- previously
    # build_belief_prompt() shuffled independently of build_response_schema(),
    # so the model had to freely cross-reference two differently-ordered
    # lists of ~40 near-identical questions purely by title-text matching,
    # which is the likely cause of the title/category mismatches described
    # in parse_structured_response().
    shuffled_features = features[:]
    random.shuffle(shuffled_features)

    prompt = build_belief_prompt(row, shuffled_features)
    response_schema = build_response_schema(shuffled_features, schema_name)

    # NOTE: no "system" message -- per Chris's review (and confirmed via
    # https://github.com/openai/harmony/blob/main/docs/format.md), the system
    # slot in gpt-oss's harmony format is meant for model-level settings
    # (identity/cutoff/reasoning effort), which we already set via separate
    # request parameters (or leave at backend default -- see reasoning_effort
    # note below). A generic "You are a helpful assistant." string doesn't
    # serve that purpose and is just redundant text, so it's dropped rather
    # than replaced. Also: this backend does NOT support a "developer" role
    # (confirmed empirically -- HTTP 422 "Input should be 'system', 'user',
    # 'assistant' or 'tool'"), so that's not an alternative here either.
    messages = [
        {"role": "user", "content": prompt},
    ]

    body = {
        "model": model_name,
        "max_tokens": max_tokens,
        "temperature": 0,
        "messages": messages,
        "response_format": {
            "type": "json_schema",
            "json_schema": response_schema,
        },
        # reasoning_effort is intentionally NOT set here -- confirmed via a
        # real test that "high" can make the model loop on repetitive
        # reasoning and burn the whole token budget with an empty final
        # answer (finish_reason: "length", content: ""). Leaving it unset
        # uses the backend's default, which behaved fine in every real call
        # made so far.
    }

    headers = {
        "Content-Type": "application/json",
        "api-key": api_key,
    }

    # timeout so one hung connection can't block a worker thread forever;
    # a timeout is a network error -- see NETWORK_ERRORS
    response = requests.post(url, headers=headers, data=json.dumps(body), timeout=300)
    if response.status_code == 429:
        raise RateLimitedError(parse_retry_after(response))
    # record the error text for alll non-2xx responses
    if not response.ok:
        error_text = response.text
        if response.status_code == 400 and "context length" in error_text.lower():
            raise ContextLengthExceededError(error_text)
        # any other non-2xx: surface the real body instead of a generic summary,
        # so future failure causes (content filter, quota, malformed request, ...)
        # are visible in the retry logs instead of just "400 Client Error: ..."
        raise RuntimeError(f"Request failed with status {response.status_code}: {error_text}")

    content_json = response.json()
    message = content_json["choices"][0]["message"]
    reply = message.get("content") # extract the structured JSON string from the response
    if reply is None:
        raise ValueError(f"Structured output returned no content: {message}")

    # Azure-specific extension field with the model's raw chain-of-thought.
    # Kept as a lightweight audit trail now that `explanation` has been
    # dropped from the schema -- lets us spot-check *why* a symbol/category
    # was chosen without paying for a separate structured `explanation`
    # field. It's free-form text covering ALL categories in this one call,
    # not broken out per category, so it's for manual/spot-check review,
    # not for programmatic per-category extraction.
    reasoning_content = message.get("reasoning_content")

    structured_payload = json.loads(reply) # convert the JSON string into a Python dictionary
    return reply, structured_payload, reasoning_content # raw content text, parsed JSON, and raw reasoning trace -- for logging/persistence and audit


# ------------------------------------------------------------
# 7. GPT call with retry logic
# ------------------------------------------------------------
# determine whether a value is considered "missing" (None or empty string)
def is_missing_value(value, check_text_quality=False):
    if value is None:
        return True
    if isinstance(value, str):
        stripped = value.strip()
        if stripped == "":
            return True
        if check_text_quality and not re.search(r"[A-Za-z]{2,}", stripped):
            return True
    return False

def infer_beliefs_with_retry(row, model_name, features, prefix=None, schema_name=None, max_retries=15):
    # returns (response_text, parsed_result, skip_reason, reasoning_content, attempts)
    # skip_reason is None on success; otherwise a short string describing why
    # parsed_result is None, e.g. "context_length_exceeded: <detail>" or
    # "exhausted_retries: <last error>" -- this is what actually gets persisted
    # into the Belief_Skip_Reason/Demo_Skip_Reason CSV columns, so the real
    # failure cause survives even if nobody reads the printed logs.
    # reasoning_content reflects the last attempt for which infer_beliefs()
    # itself returned successfully (i.e. got a parseable response) -- on a
    # ContextLengthExceededError, or if every attempt's HTTP call/JSON parse
    # failed outright, it stays None since no reasoning trace was ever received.
    # attempts is how many API calls this took (1 = first try succeeded) --
    # persisted as Belief_Attempts/Demo_Attempts so the retry rate (e.g. from
    # title/symbol mismatches) can be measured from the CSV. 429 rate-limit
    # waits, network errors and the one shrink-the-budget retry on a context
    # overflow are not counted as attempts -- none of them are model mistakes.

    # ensure schema name is always provided explicitly
    if schema_name is None:
        raise ValueError("schema_name must be provided explicitly.")

    # initialize variables to track the response and any errors
    response_text = None
    parsed_result = None
    reasoning_content = None
    last_error = None
    network_waits = 0  # consecutive network errors in this call, drives the backoff below
    max_tokens = MAX_COMPLETION_TOKENS  # shrunk at most once, on an input + completion overflow

    attempt = 0
    while attempt < max_retries:
        attempt += 1
        try:
            # get the response message in both raw text and structured JSON format
            response_text, structured_payload, reasoning_content = infer_beliefs(row, model_name, features, schema_name, max_tokens)
            network_waits = 0
            # parse the structured JSON response into a dictionary of values
            parsed_result = parse_structured_response(structured_payload, features, prefix=prefix)
            # even if the parsing succeeded, check for any missing values in the parsed result
            if any(
                is_missing_value(value)
                for value in parsed_result.values()
            ):
                 raise ValueError("Parsed structured response is incomplete.")
            
            print(f"Parsing succeeded for {row['username']}")
            return response_text, parsed_result, None, reasoning_content, attempt # return the raw response, parsed dictionary, no skip reason, the reasoning trace, and the attempt count

        except RateLimitedError as e:
            # not this row's fault -- wait as told and retry without using up an
            # attempt; the random jitter keeps threads from all retrying at once
            attempt -= 1
            record_rate_limit_hit()
            print(f"Rate limited for user {row['username']} ({model_name}); waiting {e.retry_after}s")
            time.sleep(e.retry_after + random.uniform(0, 2))

        except NETWORK_ERRORS as e:
            # not this row's fault either -- wait (15s, 30s, 60s, ... capped at
            # NETWORK_WAIT_MAX) and retry without using up an attempt, for as
            # long as the outage lasts
            attempt -= 1
            network_waits += 1
            record_network_error_hit()
            wait = min(NETWORK_WAIT_MAX, 15 * 2 ** (network_waits - 1))
            print(f"Network error for user {row['username']} ({model_name}): {type(e).__name__}; waiting {wait}s")
            time.sleep(wait + random.uniform(0, 2))

        except ContextLengthExceededError as e:
            # if the input itself fits and only input + max_tokens doesn't,
            # retry once with the completion budget shrunk to what's left --
            # as long as that's still enough for a full answer. `room <
            # max_tokens` makes this a one-time shrink, so it can't loop.
            input_tokens = parse_input_tokens(str(e))
            if input_tokens is not None:
                room = CONTEXT_LIMIT_TOKENS - input_tokens - COMPLETION_MARGIN_TOKENS
                if min_completion_tokens(features) <= room < max_tokens:
                    attempt -= 1
                    max_tokens = room
                    print(f"Input ({input_tokens} tokens) fits but not with the full completion budget for user {row['username']} ({model_name}); retrying with max_tokens={max_tokens}")
                    continue
            # otherwise (input alone too long, or too little room left for an
            # answer) log the error and skip without retrying
            print(f"Context length exceeded for user {row['username']} ({model_name}); skipping without retry: {e}")
            return "CONTEXT_LENGTH_EXCEEDED", None, f"context_length_exceeded: {e}", None, attempt

        except Exception as e:
            # for any other exception, log the error and retry with exponential backoff
            last_error = str(e)
            parsed_result = None # a failed attempt's partial parse must not count as success below
            print(f"Attempt {attempt}/{max_retries} failed for user {row['username']} ({model_name}): {e}")
            if attempt < max_retries:
                time.sleep(2 ** attempt * 0.1)

    # if all retry fails, return the last error as the skip reason
    skip_reason = f"exhausted_retries: {last_error}" if parsed_result is None else None
    return response_text, parsed_result, skip_reason, reasoning_content, max_retries # e.g., ("", None, "exhausted_retries: <last error>", None, 15)


In [ ]:
# ------------------------------------------------------------
# Pilot: pick ~30 users to run through the production loop first
# ------------------------------------------------------------
# The production cell below runs these users (RUN_USER_IDS = pilot_user_ids)
# and writes them into the real all_results_7d.csv. The full run later skips
# them by (id, time_window_start), so as long as prompt/schema/params don't
# change after the pilot, pilot rows never need to be re-run.
#
# Groups:
#   over_limit  -- 3 users with at least one window past the context limit
#                  (checks the context_length_exceeded skip path end to end)
#   long_prompt -- 7 users with the longest windows still under the limit
#   political   -- 10 users with the densest political-keyword tweets
#                  (most likely to have real signal on all 40 belief items)
#   random      -- 10 random users from the rest
CONTEXT_CHAR_LIMIT = 400_000  # ~104K tokens at 3.85 chars/token, see 4.flag_context_length_risk.py
POLITICAL_RE = r"(?i)\b(?:trump|biden|harris|kamala|maga|democrats?|republicans?|gop|immigra\w*|border|abortion|guns?|election|vote|voting)\b"
PILOT_SEED = 42

nonempty = merged_df[~merged_df["is_empty_window"]].copy()
nonempty["tweet_chars"] = nonempty["tweets_struct"].fillna("").str.len()
nonempty["political_hits"] = nonempty["tweets_struct"].fillna("").str.count(POLITICAL_RE)

user_stats = nonempty.groupby("id").agg(
    n_windows=("tweet_chars", "size"),
    max_chars=("tweet_chars", "max"),
    total_chars=("tweet_chars", "sum"),
    political_hits=("political_hits", "sum"),
)
user_stats["political_density"] = user_stats["political_hits"] / user_stats["total_chars"].clip(lower=1)

over_limit = user_stats[user_stats["max_chars"] > CONTEXT_CHAR_LIMIT]
under_limit = user_stats[user_stats["max_chars"] <= CONTEXT_CHAR_LIMIT]

over_limit_ids = over_limit.sample(n=min(3, len(over_limit)), random_state=PILOT_SEED).index
long_prompt_ids = under_limit["max_chars"].nlargest(7).index
remaining = user_stats.drop(over_limit_ids.union(long_prompt_ids))
political_ids = remaining[remaining["n_windows"] >= 10]["political_density"].nlargest(10).index
remaining = remaining.drop(political_ids)
random_ids = remaining.sample(n=10, random_state=PILOT_SEED).index

pilot_groups = pd.concat([
    pd.Series("over_limit", index=over_limit_ids),
    pd.Series("long_prompt", index=long_prompt_ids),
    pd.Series("political", index=political_ids),
    pd.Series("random", index=random_ids),
])
pilot_user_ids = list(pilot_groups.index)

pilot_nonempty_rows = nonempty["id"].isin(pilot_user_ids).sum()
print(f"Pilot: {len(pilot_user_ids)} users, {pilot_nonempty_rows} non-empty windows "
      f"({pilot_nonempty_rows / len(nonempty):.1%} of all non-empty windows)")
user_stats.loc[pilot_user_ids].assign(group=pilot_groups)


In [ ]:
# ------------------------------------------------------------
# 8. Apply inference to the dataframe
# ------------------------------------------------------------
# NOTE: this is the real production loop -- it makes real API calls and
# writes to the real all_results_7d.csv / users_7d/. Split into its own
# cell on purpose so it can never be triggered accidentally by running the
# helper-definitions cell above it.
#
# Which users it runs is controlled by RUN_USER_IDS: set it to
# pilot_user_ids (from the pilot cell above) for the pilot, or None for the
# full merged_df. Pilot rows land in the same output file as the full run.
#
# Resuming is by (id, time_window_start) key, not by checkpoint row index:
# any row already in all_results_7d.csv is skipped, everything else is run.
# That is what lets the full run skip the pilot users, and it also means a
# row that raised (and so was never written) gets picked up again just by
# re-running this cell -- re-run it once after the full run finishes to fill
# in any such rows.
#
# Parallelism: MAX_WORKERS threads, one *user* per task -- each user's
# windows run in order inside one thread, so users_7d/{id}.csv is only ever
# written by that thread; only the shared all_results_7d.csv needs a lock.
# The pilot has 30 users, so more than 30 workers can't be used there.
# Interrupting the kernel lets in-flight rows finish, cancels users not yet
# started, and is safe to resume from by re-running this cell.
from concurrent.futures import ThreadPoolExecutor, as_completed

RUN_USER_IDS = None  # None = full run over all of merged_df
MAX_WORKERS = 50  # pilot: one per pilot user; full run: start at 30-50 (5M TPM / 5K RPM deployment)

models = [
    "gpt-oss-120b"
]

df = merged_df if RUN_USER_IDS is None else merged_df[merged_df["id"].isin(RUN_USER_IDS)]

output_file = "../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/all_results_7d.csv"
user_dir = "../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/users_7d"
os.makedirs(user_dir, exist_ok=True)

# fixed, complete set of output columns computed up front -- append-mode writing
# needs a header that never changes once writing starts, so this can't be
# inferred lazily from whatever keys happen to be in the first row.
# *_Seconds = wall-clock time of the whole infer_beliefs_with_retry() call,
# including retries, backoff, rate-limit and network waits -- i.e. what a row
# actually costs in run time, used to size the full run / parallelism from the pilot.
output_columns = list(merged_df.columns) + ["Model", "Belief_Raw", "Demo_Raw", "Belief_Skip_Reason", "Demo_Skip_Reason", "Belief_Reasoning", "Demo_Reasoning", "Belief_Attempts", "Demo_Attempts", "Belief_Seconds", "Demo_Seconds"]
for feature in belief_features:
    key = feature["id"]
    output_columns += [
        f"Belief_{key}_Category",
        f"Belief_{key}_Symbol", f"Belief_{key}_Speculation",
    ]
for feature in demo_features:
    key = feature["id"]
    output_columns += [
        f"Demo_{key}_Category",
        f"Demo_{key}_Symbol", f"Demo_{key}_Speculation",
    ]

file_is_new = not os.path.exists(output_file)

# rows already written in a previous run (pilot or interrupted full run)
done_keys = set()
if not file_is_new:
    done_df = pd.read_csv(output_file, usecols=["id", "time_window_start"], encoding="utf-8-sig")
    done_keys = set(zip(done_df["id"], done_df["time_window_start"]))
print(f"Resuming: {len(done_keys)} rows already in {output_file}.")
print(f"This run covers {df['id'].nunique()} users / {len(df)} rows, {MAX_WORKERS} workers.")

output_fh = open(output_file, "a", newline="", encoding="utf-8-sig")
output_writer = csv.DictWriter(output_fh, fieldnames=output_columns)
if file_is_new:
    output_writer.writeheader()

write_lock = threading.Lock()     # guards output_writer / output_fh
progress_lock = threading.Lock()  # guards processed_rows
stop_event = threading.Event()    # set on interrupt -- workers stop before their next row

completed_users = set()
total_users = df['id'].nunique()
processed_rows = 0
rate_limit_hits = 0     # incremented by record_rate_limit_hit() in the helper cell
network_error_hits = 0  # incremented by record_network_error_hit() in the helper cell


def build_empty_row(row, model_name):
    # empty windows skip the API calls entirely -- all outputs are None
    combined_row = row.to_dict()
    combined_row.update({
        "Model": model_name,
        "Belief_Raw": None,
        "Demo_Raw": None,
        "Belief_Reasoning": None,
        "Demo_Reasoning": None,
        "Belief_Attempts": None,
        "Demo_Attempts": None,
        "Belief_Seconds": None,
        "Demo_Seconds": None,
    })
    for feature in belief_features:
        key = feature["id"]
        combined_row.update({
            f"Belief_{key}_Category": None,
            f"Belief_{key}_Symbol": None,
            f"Belief_{key}_Speculation": None,
        })
    for feature in demo_features:
        key = feature["id"]
        combined_row.update({
            f"Demo_{key}_Category": None,
            f"Demo_{key}_Symbol": None,
            f"Demo_{key}_Speculation": None,
        })
    return combined_row


def run_one_row(row, model_name):
    if row.get("is_empty_window", False):
        return build_empty_row(row, model_name)

    combined_row = row.to_dict()

    belief_start = time.time()
    belief_text, parsed_belief, belief_skip_reason, belief_reasoning, belief_attempts = infer_beliefs_with_retry(
        row,
        model_name,
        belief_features,
        prefix="Belief",
        schema_name="belief_assessments",
    )
    belief_seconds = round(time.time() - belief_start, 1)

    demo_start = time.time()
    demo_text, parsed_demo, demo_skip_reason, demo_reasoning, demo_attempts = infer_beliefs_with_retry(
        row,
        model_name,
        demo_features,
        prefix="Demo",
        schema_name="demo_assessments",
    )
    demo_seconds = round(time.time() - demo_start, 1)

    combined_row.update({
        "Model": model_name,
        "Belief_Raw": belief_text,
        "Demo_Raw": demo_text,
        "Belief_Skip_Reason": belief_skip_reason,
        "Demo_Skip_Reason": demo_skip_reason,
        "Belief_Reasoning": belief_reasoning,
        "Demo_Reasoning": demo_reasoning,
        "Belief_Attempts": belief_attempts,
        "Demo_Attempts": demo_attempts,
        "Belief_Seconds": belief_seconds,
        "Demo_Seconds": demo_seconds,
    })
    combined_row.update(parsed_belief or {})
    combined_row.update(parsed_demo or {})
    return combined_row


def process_user(user_id, user_rows, model_name):
    # runs all of one user's windows in order, inside one worker thread
    global processed_rows

    user_file = f"{user_dir}/{user_id}.csv"
    if os.path.exists(user_file):
        user_results = pd.read_csv(user_file, encoding="utf-8-sig").to_dict("records")
    else:
        user_results = []

    for index, row in user_rows.iterrows():
        if stop_event.is_set():
            break
        if (row["id"], row["time_window_start"]) in done_keys:
            continue
        try:
            combined_row = run_one_row(row, model_name)

            with write_lock:
                output_writer.writerow(combined_row)
                output_fh.flush()

            user_results.append(combined_row)
            pd.DataFrame(user_results).to_csv(user_file, index=False, encoding="utf-8-sig")

            if not row.get("is_empty_window", False):
                with progress_lock:
                    processed_rows += 1
                    if processed_rows % 50 == 0:
                        print(f"Progess: {processed_rows} non-empty rows processed this run")
        except Exception as e:
            print(f"Error processing row {index} for model {model_name}: {e}")

    return user_id


for model_name in models:
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    futures = {
        pool.submit(process_user, user_id, user_rows, model_name): user_id
        for user_id, user_rows in df.groupby("id", sort=False)
    }
    try:
        for future in as_completed(futures):
            user_id = futures[future]
            try:
                future.result()
                completed_users.add(user_id)
                print(f"Finished all timewindows for user {user_id} -- {len(completed_users)}/{total_users} users completed")
            except Exception as e:
                print(f"User {user_id} failed: {e}")
    except KeyboardInterrupt:
        print("Interrupted -- letting in-flight rows finish, then stopping. Re-run this cell to resume.")
        stop_event.set()
        pool.shutdown(wait=True, cancel_futures=True)
        break
    finally:
        pool.shutdown(wait=True)

output_fh.close()
print(f"Done! {processed_rows} non-empty rows processed this run, "
      f"{rate_limit_hits} rate-limit (429) waits, {network_error_hits} network-error waits.")


In [ ]:
# ------------------------------------------------------------
# Pilot summary: retry rate, skip rate, and time per call
# ------------------------------------------------------------
# Attempts = API calls per row (1 = first try succeeded, 15 = exhausted).
# Skip_Reason is empty on success, otherwise "context_length_exceeded: ..."
# or "exhausted_retries: ...".
# Seconds = wall-clock time per call including retries.
pilot_results = pd.read_csv(
    output_file,
    usecols=["id", "time_window_start", "is_empty_window",
             "Belief_Attempts", "Demo_Attempts", "Belief_Skip_Reason", "Demo_Skip_Reason",
             "Belief_Seconds", "Demo_Seconds"],
    encoding="utf-8-sig",
)
pilot_results = pilot_results[pilot_results["id"].isin(pilot_user_ids) & ~pilot_results["is_empty_window"]].copy()
pilot_results["group"] = pilot_results["id"].map(pilot_groups)
print(f"Pilot non-empty rows written: {len(pilot_results)} / expected {pilot_nonempty_rows}")

for prefix in ["Belief", "Demo"]:
    attempts = pilot_results[f"{prefix}_Attempts"]
    seconds = pilot_results[f"{prefix}_Seconds"]
    outcome = pilot_results[f"{prefix}_Skip_Reason"].fillna("ok").str.split(":").str[0]
    print(f"\n{prefix}: mean attempts {attempts.mean():.2f}, rows needing a retry {(attempts > 1).mean():.1%}")
    print(f"  attempts distribution: {attempts.value_counts().sort_index().to_dict()}")
    print(f"  outcome: {outcome.value_counts().to_dict()}")
    print(f"  seconds per call: median {seconds.median():.1f}, mean {seconds.mean():.1f}, p90 {seconds.quantile(0.9):.1f}")

# rough single-threaded estimate for the full run, from the pilot's mean
# time per non-empty row (+1s sleep per row in the loop)
sec_per_row = (pilot_results["Belief_Seconds"] + pilot_results["Demo_Seconds"]).mean() + 1
total_nonempty = (~merged_df["is_empty_window"]).sum()
print(f"\nMean {sec_per_row:.1f}s per non-empty row -> full run of {total_nonempty} rows "
      f"~{sec_per_row * total_nonempty / 3600:.0f} hours single-threaded")

pilot_results.assign(
    belief_skipped=pilot_results["Belief_Skip_Reason"].notna(),
    demo_skipped=pilot_results["Demo_Skip_Reason"].notna(),
).groupby("group").agg(
    rows=("id", "size"),
    belief_mean_attempts=("Belief_Attempts", "mean"),
    demo_mean_attempts=("Demo_Attempts", "mean"),
    belief_skip_rate=("belief_skipped", "mean"),
    demo_skip_rate=("demo_skipped", "mean"),
    belief_mean_seconds=("Belief_Seconds", "mean"),
    demo_mean_seconds=("Demo_Seconds", "mean"),
)
